# **Tratamento de Dados e Pré-processamento**

Para este problema de classificação, o objetivo é prever o resultado da partida: Vitória do Mandante, Empate ou Vitória do Visitante.
A variável alvo (resultado) foi derivada comparando a coluna vencedor com a coluna mandante.
Como os algoritmos de aprendizado de máquina operam com cálculos matemáticos, eles não conseguem interpretar texto diretamente (como "Flamengo" ou "Maracanã"). Portanto, aplicou-se o método One-Hot Encoding nas variáveis categóricas independentes (mandante, visitante, estado). Essa técnica cria colunas binárias (0 ou 1) para cada categoria, eliminando qualquer viés de ordem ou hierarquia que algoritmos baseados em distância (como KNN) poderiam inferir erroneamente.
Todo esse processo foi encapsulado em um Pipeline junto com um ColumnTransformer para evitar vazamento de dados (data leakage) durante a validação cruzada.

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import kagglehub
import os


path = kagglehub.dataset_download("adaoduque/campeonato-brasileiro-de-futebol")
print("Pasta base do Kaggle:", path)
caminho_csv = os.path.join(path, "campeonato-brasileiro-full.csv")
dados = pd.read_csv(caminho_csv)


In [ ]:
dados.head(3)

In [ ]:
print(f"Quantidade de partidas registradas no dataset: {len(dados)}")

# **Tratamento de Dados e Valores Ausentes (Missing Values)**
Durante a análise exploratória, identificou-se uma alta incidência de valores ausentes em colunas como tecnico_mandante, tecnico_visitante e formações táticas, devido à falta de registros históricos nas primeiras décadas do campeonato (a partir de 2003).

A estratégia de tratamento adotada foi:

As colunas com mais de 50% de dados faltantes (técnicos, formações e arrecadação) foram removidas do modelo, pois sua imputação causaria forte viés e sua ausência prejudicaria o aprendizado.

Para as variáveis essenciais (mandante, visitante, vencedor e Estado), caso houvesse apenas alguns valores nulos, aplicaria-se a exclusão das linhas (dropna()), garantindo que o algoritmo (Regressão Logística/KNN/SVM) recebesse uma matriz densa e consistente para o treinamento.

In [ ]:
print("Valores nulos antes do tratamento:")
print(dados.isnull().sum())

In [ ]:
colunas_para_remover = ['formacao_mandante', 'formacao_visitante', 'tecnico_mandante', 'tecnico_visitante', 'arrecadacao']

In [ ]:
dados = dados.drop(columns=colunas_para_remover)

In [ ]:
print("Valores nulos após o tratamento:")
print(dados.isnull().sum())

# **Modelo 1: Regressão Logística**

In [ ]:
dados['resultado'] = np.where(dados['vencedor'] == dados['mandante'], 'Mandante',
                    np.where(dados['vencedor'] == '-', 'Empate', 'Visitante'))

In [ ]:
features_categoricas = ['mandante', 'visitante', 'mandante_Estado', 'visitante_Estado']

In [ ]:
X = dados[features_categoricas]
y = dados['resultado']

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
pre_processador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), features_categoricas)
    ])

In [ ]:
pipeline = Pipeline([
    ('pre_processamento', pre_processador),
    ('classificador', LogisticRegression(max_iter=1000))
])

In [ ]:
parametros = {
    'classificador__C': [0.1, 1.0, 10.0],
    'classificador__solver': ['lbfgs', 'liblinear']
}

In [ ]:

grid_search = GridSearchCV(pipeline, param_grid=parametros, cv=5, scoring='accuracy', n_jobs=-1)

In [ ]:
#Treinamento da Regressão Logística
grid_search.fit(X_treino, y_treino)

In [ ]:
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor acurácia na validação cruzada: {grid_search.best_score_:.4f}")

In [ ]:
previsoes = grid_search.predict(X_teste)
print("\nRelatório de Classificação Final:")
print(classification_report(y_teste, previsoes))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# **Modelo 2: K-Nearest Neighbors (KNN)**

In [ ]:
pipeline_knn = Pipeline([
    ('pre_processamento', pre_processador),
    ('classificador', KNeighborsClassifier())
])

In [ ]:
parametros_knn = {
    'classificador__n_neighbors': [3, 5, 7],
    'classificador__weights': ['uniform', 'distance']
}

In [ ]:
#Treinamento do KNN
grid_knn = GridSearchCV(pipeline_knn, param_grid=parametros_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_treino, y_treino)

In [ ]:
print(f"Melhores parâmetros KNN: {grid_knn.best_params_}")
print(f"Melhor acurácia (Validação Cruzada): {grid_knn.best_score_:.4f}")

In [ ]:
previsoes_knn = grid_knn.predict(X_teste)

In [ ]:
#Relatório de Classificação (KNN):
print(classification_report(y_teste, previsoes_knn))

# **Modelo 3: Árvore de Decisão**

In [ ]:
pipeline_tree = Pipeline([
    ('pre_processamento', pre_processador),
    ('classificador', DecisionTreeClassifier(random_state=42))
])

In [ ]:
parametros_tree = {
    'classificador__max_depth': [None, 10, 20],
    'classificador__min_samples_split': [2, 5, 10]
}

In [ ]:
#Treinamento da Árvore de Decisão
grid_tree = GridSearchCV(pipeline_tree, param_grid=parametros_tree, cv=5, scoring='accuracy', n_jobs=-1)
grid_tree.fit(X_treino, y_treino)

In [ ]:
print(f"Melhores parâmetros Árvore: {grid_tree.best_params_}")
print(f"Melhor acurácia (Validação Cruzada): {grid_tree.best_score_:.4f}")

In [ ]:
previsoes_tree = grid_tree.predict(X_teste)
print("\nRelatório de Classificação (Árvore de Decisão):")
print(classification_report(y_teste, previsoes_tree))

# **Modelo 4: Multilayer Perceptron (MLP)**

In [ ]:
pipeline_mlp = Pipeline([
    ('pre_processamento', pre_processador),
    ('classificador', MLPClassifier(max_iter=500, random_state=42))
])

In [ ]:
parametros_mlp = {
    'classificador__hidden_layer_sizes': [(50,), (100,)],
    'classificador__activation': ['relu', 'tanh']
}

In [ ]:
#Treinamento do MLP
grid_mlp = GridSearchCV(pipeline_mlp, param_grid=parametros_mlp, cv=5, scoring='accuracy', n_jobs=-1)
grid_mlp.fit(X_treino, y_treino)

In [ ]:
print(f"Melhores parâmetros MLP: {grid_mlp.best_params_}")
print(f"Melhor acurácia (Validação Cruzada): {grid_mlp.best_score_:.4f}")

In [ ]:
previsoes_mlp = grid_mlp.predict(X_teste)
print("\nRelatório de Classificação (MLP):")
print(classification_report(y_teste, previsoes_mlp))

# **Modelo 5: Support Vector Machine (SVM)**

In [ ]:
pipeline_svm = Pipeline([
    ('pre_processamento', pre_processador),
    ('classificador', SVC(random_state=42))
])

In [ ]:
parametros_svm = {
    'classificador__kernel': ['linear', 'rbf'],
    'classificador__C': [0.1, 1.0]
}

In [ ]:
#Treinamento do SVM
grid_svm = GridSearchCV(pipeline_svm, param_grid=parametros_svm, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_treino, y_treino)

In [ ]:
print(f"Melhores parâmetros SVM: {grid_svm.best_params_}")
print(f"Melhor acurácia (Validação Cruzada): {grid_svm.best_score_:.4f}")

In [ ]:
previsoes_svm = grid_svm.predict(X_teste)
print("\nRelatório de Classificação (SVM):")
print(classification_report(y_teste, previsoes_svm))

# **Comparação Entre os Modelos Treinados**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
acuracias = {
    'Regressão Logística': grid_search.score(X_teste, y_teste),
    'KNN': grid_knn.score(X_teste, y_teste),
    'Árvore de Decisão': grid_tree.score(X_teste, y_teste),
    'MLP (Redes Neurais)': grid_mlp.score(X_teste, y_teste),
    'SVM': grid_svm.score(X_teste, y_teste)
}

In [ ]:
plt.figure(figsize=(12, 6))
grafico = sns.barplot(x=list(acuracias.keys()), y=list(acuracias.values()), palette='viridis')

plt.title('Comparação de Acurácia entre os Modelos de Classificação', fontsize=14, pad=15)
plt.ylabel('Acurácia no Conjunto de Teste', fontsize=12)
plt.ylim(0, 1)

for i, v in enumerate(acuracias.values()):
    grafico.text(i, v + 0.01, f"{v:.4f}", ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
modelos_treinados = {
    'Regressão Logística': grid_search,
    'KNN': grid_knn,
    'Árvore de Decisão': grid_tree,
    'MLP (Redes Neurais)': grid_mlp,
    'SVM': grid_svm
}

# Cria uma figura grande com uma grade de 2 linhas e 3 colunas
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
fig.suptitle('Matrizes de Confusão - Comparação de Todos os Modelos', fontsize=16, fontweight='bold', y=1.02)

# Achata a matriz de eixos para facilitar o loop
axes = axes.flatten()

# Classes no formato correto
classes_nomes = grid_search.classes_

# Loop para plotar cada matriz em um "quadradinho" da grade
for ax, (nome, modelo) in zip(axes, modelos_treinados.items()):
    # Faz a previsão com o modelo atual
    previsoes_atuais = modelo.predict(X_teste)

    # Calcula a matriz
    cm = confusion_matrix(y_teste, previsoes_atuais, labels=classes_nomes)

    # Plota o mapa de calor
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=classes_nomes, yticklabels=classes_nomes)

    ax.set_title(nome, fontsize=12, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')

axes[5].axis('off')

plt.tight_layout()
plt.show()